# Load Libraries

In [1]:
import os
import warnings
import logging
import sys
import pickle
import numpy as np
import pandas as pd
import geopandas as gpd
import dotenv
import pyet
import matplotlib.pyplot as plt


# Load environment variables from .env file
dotenv.load_dotenv()

# Set up logging
logging.basicConfig(level=logging.INFO)

# Suppress warnings
warnings.filterwarnings("ignore")

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', None)

# Load Data

In [2]:
# Load Parquet Data
data = pd.read_parquet('../../data/Iran_Monthly_ETo_1951_2025.parquet')
logging.info("Data loaded from parquet file.")    

INFO:root:Data loaded from parquet file.


In [3]:
data

,year,month,region_id,region_name,station_id,station_name,lat,lon,station_elevation,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,date,FAO56,Hargreaves,Blaney_Criddle,Oudin
0,1951,1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-01-01,NaN,NaN,NaN,NaN
1,1951,2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-02-01,NaN,NaN,NaN,NaN
2,1951,3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-03-01,NaN,NaN,NaN,NaN
3,1951,4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-04-01,NaN,NaN,NaN,NaN
4,1951,5,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,1951-05-01,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688891,2025,5,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,2025-05-01,NaN,147.74,110.63,121.41
688892,2025,6,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,2025-06-01,NaN,207.46,125.91,138.41
688893,2025,7,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,2025-07-01,NaN,232.24,149.49,166.81
688894,2025,8,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,2025-08-01,NaN,213.22,145.58,162.03


# Data Description

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 688896 entries, 0 to 688895
Data columns (total 32 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   year               688896 non-null  int32         
 1   month              688896 non-null  int32         
 2   region_id          688896 non-null  object        
 3   region_name        688896 non-null  object        
 4   station_id         688896 non-null  object        
 5   station_name       688896 non-null  object        
 6   lat                688896 non-null  float64       
 7   lon                688896 non-null  float64       
 8   station_elevation  688896 non-null  float64       
 9   tmax               162530 non-null  float64       
 10  tmax_count         688896 non-null  int64         
 11  tmin               161780 non-null  float64       
 12  tmin_count         688896 non-null  int64         
 13  tm                 161568 non-null  float64 

In [5]:
data = data[[
    'region_id', 'region_name', 'station_id', 'station_name',
    'lat', 'lon', 'station_elevation',
    'date', 'year', 'month',
    'tmax', 'tmax_count', 'tmin', 'tmin_count', 'tm', 'tm_count', 
    'umax', 'umax_count', 'umin', 'umin_count', 'um', 'um_count',
    'ffm', 'ffm_count', 'sshn', 'sshn_count', 'rrr24', 'rrr24_count',
    'FAO56', 'Hargreaves', 'Blaney_Criddle', 'Oudin'
]]

data

,region_id,region_name,station_id,station_name,lat,lon,station_elevation,date,year,month,tmax,tmax_count,tmin,tmin_count,tm,tm_count,umax,umax_count,umin,umin_count,um,um_count,ffm,ffm_count,sshn,sshn_count,rrr24,rrr24_count,FAO56,Hargreaves,Blaney_Criddle,Oudin
0,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-01-01,1951,1,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
1,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-02-01,1951,2,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
2,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-03-01,1951,3,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
3,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-04-01,1951,4,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
4,OIID,Airforce,40753,Dowshan Tappeh,35.70,51.48,1209.20,1951-05-01,1951,5,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
688891,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-05-01,2025,5,24.46,13,12.53,10,19.07,10,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,12.20,11,NaN,147.74,110.63,121.41
688892,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-06-01,2025,6,32.03,30,12.43,28,22.12,28,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,29,NaN,207.46,125.91,138.41
688893,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-07-01,2025,7,36.54,31,17.80,30,27.14,30,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,3.10,30,NaN,232.24,149.49,166.81
688894,OITZ,Zanjan,18699,Zarrenrood,35.75,48.48,1751.00,2025-08-01,2025,8,37.20,20,19.91,11,29.04,11,NaN,0,NaN,0,NaN,0,NaN,0,NaN,0,0.00,19,NaN,213.22,145.58,162.03


In [6]:
print(f"Number of unique regions: {data.region_name.nunique()}")
print(f"Region names: {data.region_name.unique().tolist()}")

Number of unique regions: 32
Region names: ['Airforce', 'Alborz', 'Ardebil', 'Azarbayjan-E-Gharbi', 'Azarbayjan-E-Sharghi', 'Bushehr', 'Chaharmahal Va Bakhtiari', 'Esfahan', 'Fars', 'Gilan', 'Golestan', 'Hamedan', 'Hormozgan', 'Ilam', 'Kerman', 'Kermanshah', 'Khohgiluyeh Va Boyerahmad', 'Khorasan Razavi', 'Khuzestan', 'Kordestan', 'Lorestan', 'Markazi', 'Mazandaran', 'North Khorasan', 'Qazvin', 'Qom', 'Semnan', 'Sistan Va Baluchestan', 'South Khorasan', 'Tehran', 'Yazd', 'Zanjan']


In [9]:
d = data[['region_id', 'region_name', 'station_name', 'station_id', 'lat', 'lon', 'station_elevation']].drop_duplicates().reset_index(drop=True)
d[d.duplicated(subset=['region_name', 'station_name', 'station_id'], keep=False)]

,region_id,region_name,station_name,station_id,lat,lon,station_elevation


In [12]:
print(f"Number of unique stations:")
data[['region_name', 'station_name', 'station_id']].drop_duplicates().reset_index(drop=True)

Number of unique stations:


,region_name,station_name,station_id
0,Airforce,Dowshan Tappeh,40753
1,Airforce,Hamedan (Nozheh),40767
2,Airforce,Khurbirjand,99437
3,Airforce,Konarak (Airport),40897
4,Alborz,Asara,18590
...,...,...,...
763,Zanjan,Soltaniyeh,18451
764,Zanjan,Zanjan,40729
765,Zanjan,Zanjan (Airport),88118
766,Zanjan,Zarinabad(Egrood),18454


In [ ]:
# change *_count of days to percent base on number of days in month, year
def days_in_month(year, month):
    if month == 2:
        if (year % 4 == 0 and year % 100 != 0) or (year % 400 == 0):
            return 29
        else:
            return 28
    elif month in [4, 6, 9, 11]:
        return 30
    else:
        return 31

data['days_in_month'] = data.apply(lambda row: days_in_month(row['year'], row['month']), axis=1)

for col in data.columns:
    if col.endswith('_count'):
        percent_col = col.replace('_count', '_percent')
        data[percent_col] = round((data[col] / data['days_in_month']) * 100, 1)


In [ ]:
data

In [ ]:
data.dropna(subset=['Hargreaves'])

In [ ]:
# filter row with *_percent greater than 80% except rrr24_percent
for col in data.columns:
    if col.endswith('_percent') and col != 'rrr24_percent':
        df = data[data[col] >= 75]
        
df

In [ ]:
# df.query("region_name == ['Khorasan Razavi', 'South Khorasan', 'North Khorasan']").groupby("station_name")["station_name"].count().sort_values(ascending=False).div(12).reset_index(name="count_years")
df.groupby(["region_name", "station_name", "station_id"])["station_name"].count().sort_values(ascending=False).div(12).reset_index(name="count_years")

In [ ]:
data.query("station_name == 'Golbaf' and region_name == 'Kerman'")